In [0]:
dbutils.secrets.list(scope="db-scope")

[SecretMetadata(key='clientid-db'),
 SecretMetadata(key='secretid'),
 SecretMetadata(key='tenantid')]

In [0]:
clientid = dbutils.secrets.get(scope="db-scope", key="clientid-db") 
clientsecret = dbutils.secrets.get(scope="db-scope", key="secretid")
tenantid = dbutils.secrets.get(scope="db-scope", key="tenantid")

In [0]:
mount_point = "/mnt/newsrecsysdata"
if any(mount.mountPoint == mount_point for mount in dbutils.fs.mounts()):
    dbutils.fs.unmount(mount_point)

configs = { 
           "fs.azure.account.auth.type": "OAuth", 
           "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider", 
           "fs.azure.account.oauth2.client.id": clientid, 
           "fs.azure.account.oauth2.client.secret": clientsecret, 
           "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{tenantid}/oauth2/token" }

dbutils.fs.mount(
    source="abfss://mind-data@mindnewsrecsystem.dfs.core.windows.net",
    mount_point=mount_point,
    extra_configs=configs
)

/mnt/newsrecsysdata has been unmounted.


True

In [0]:
dbutils.fs.ls("mnt/newsrecsysdata")

[FileInfo(path='dbfs:/mnt/newsrecsysdata/raw-data/', name='raw-data/', size=0, modificationTime=1735104264000),
 FileInfo(path='dbfs:/mnt/newsrecsysdata/transformed-data/', name='transformed-data/', size=0, modificationTime=1735104274000)]

In [0]:
from pyspark.sql import SparkSession
import numpy as np
from pyspark.sql.functions import coalesce, concat, regexp_replace, split, col, udf, trim, unix_timestamp, date_format, when, concat_ws
from pyspark.sql.types import ArrayType, StructType, StructField, StringType, IntegerType
from pyspark.sql import functions as F
import ast

In [0]:
behaviors = spark.read.csv(
    "/mnt/newsrecsysdata/raw-data/behaviors.tsv",
    sep='\t', 
    inferSchema=True, 
    header=False
).toDF('Impression ID', 'User ID', 'Timestamp', 'History', 'Impressions')

display(behaviors)

Impression ID,User ID,Timestamp,History,Impressions
1,U13740,11/11/2019 9:05:58 AM,N55189 N42782 N34694 N45794 N18445 N63302 N10414 N19347 N31801,N55689-1 N35729-0
2,U91836,11/12/2019 6:11:30 PM,N31739 N6072 N63045 N23979 N35656 N43353 N8129 N1569 N17686 N13008 N21623 N6233 N14340 N48031 N62285 N44383 N23061 N16290 N6244 N45099 N58715 N59049 N7023 N50528 N42704 N46082 N8275 N15710 N59026 N8429 N30867 N56514 N19709 N31402 N31741 N54889 N9798 N62612 N2663 N16617 N6087 N13231 N63317 N61388 N59359 N51163 N30698 N34567 N54225 N32852 N55833 N64467 N3142 N13912 N29802 N44462 N29948 N4486 N5398 N14761 N47020 N65112 N31699 N37159 N61101 N14761 N3433 N10438 N61355 N21164 N22976 N2511 N48390 N58224 N48742 N35458 N24611 N37509 N21773 N41011 N19041 N25785,N20678-0 N39317-0 N58114-0 N20495-0 N42977-0 N22407-0 N14592-0 N17059-1 N33677-0 N7821-0 N6890-0
3,U73700,11/14/2019 7:01:48 AM,N10732 N25792 N7563 N21087 N41087 N5445 N60384 N46616 N52500 N33164 N47289 N24233 N62058 N26378 N49475 N18870,N50014-0 N23877-0 N35389-0 N49712-0 N16844-0 N59685-0 N23814-1 N23446-0 N64174-0 N11817-0 N60550-0 N48225-0 N45509-0 N56711-0 N46821-0 N48017-0 N8015-0 N5364-0 N48722-0 N55555-0 N37348-0 N40109-0 N59495-0 N36226-0 N38779-0 N47346-0 N48875-0 N10960-0 N29739-0 N50872-0 N50592-0 N13131-0 N3839-0 N12330-0 N47098-0 N51570-0
4,U34670,11/11/2019 5:28:05 AM,N45729 N2203 N871 N53880 N41375 N43142 N33013 N29757 N31825 N51891,N35729-0 N33632-0 N49685-1 N27581-0
5,U8125,11/12/2019 4:11:21 PM,N10078 N56514 N14904 N33740,N39985-0 N36050-0 N16096-0 N8400-1 N22407-0 N60408-0 N61497-0 N47412-0 N41220-0 N1940-0 N724-0 N11363-0 N261-0 N33883-0 N36807-0 N11967-0 N17896-0 N13486-0 N10413-0 N54274-0 N4247-0 N27497-0 N38512-0 N30253-0 N45389-0 N20015-0 N20678-0 N54003-0 N35850-0 N33261-0 N32010-0 N57426-0 N7419-0 N50023-0 N36446-0 N26940-0 N28495-0 N19318-0 N4936-0 N28414-0 N25108-0 N32791-0 N23563-0 N39317-0 N16166-0 N37058-0 N64851-0 N46992-0 N57327-0 N12995-0 N58363-0 N53084-0 N11094-0 N36436-0 N305-0 N58241-0 N33212-0 N6975-0 N58114-0 N3344-0 N25406-0 N4741-0 N33885-0 N20915-0 N44941-0 N57319-0 N36532-0 N61822-0 N20527-0
6,U19739,11/11/2019 6:52:13 PM,N39074 N14343 N32607 N32320 N22007 N442 N19001 N24294 N51188 N22772 N51188 N12603 N8275 N19741 N6695 N35820 N30531 N15545 N27529 N62703 N59426 N15414 N54827 N21395 N39941 N10824 N42512 N58521 N62846 N14385 N47020 N2142 N17099 N47020 N11804 N52121,N21119-1 N53696-0 N33619-1 N25722-0 N2869-0
7,U8355,11/11/2019 12:22:09 PM,N8419 N15771 N1431 N5888 N18663 N24123 N22130 N20286 N32095 N46868 N55310 N31931 N34399 N42526 N64562 N12194 N23887 N56541 N59704 N30531 N60388 N8569 N38562 N47791 N157 N306 N30160 N41797 N47482 N2606 N20886 N47054 N64631 N3933 N40509,N51346-0 N33848-0 N15132-0 N10688-0 N6342-0 N61359-0 N7809-0 N64397-0 N27079-0 N47149-0 N39844-0 N52585-0 N58572-0 N15830-0 N41774-0 N3697-0 N33964-0 N54988-0 N47229-0 N9271-0 N63550-0 N40530-0 N14652-0 N51896-0 N57385-0 N49375-0 N21882-0 N14780-0 N58030-0 N30108-0 N26286-0 N9139-0 N21707-0 N45456-0 N21119-0 N45616-0 N47747-0 N36712-0 N26331-0 N46882-0 N21428-0 N41881-0 N43432-0 N57987-0 N26130-0 N3541-0 N18708-0 N12800-0 N44818-0 N6379-0 N49777-0 N2952-0 N61549-0 N63273-0 N42144-0 N3474-0 N35671-0 N58710-0 N24180-0 N6099-0 N35738-0 N55204-1 N46845-0 N41172-0 N5370-0 N63319-0 N39587-0 N13816-0 N13259-0 N8061-0 N51241-0 N20811-0 N51587-0 N35737-0 N22664-0 N12028-0 N64482-0 N20018-0 N4936-0 N22442-0 N53237-0 N15260-0 N53054-0 N41578-0 N58499-0 N29715-0 N40431-0 N23254-0 N23174-0 N1914-0 N10464-0 N154-0 N5075-0 N36621-0 N28983-0 N56598-0 N60826-0 N40065-0 N36933-0 N40725-0 N29128-0 N38155-0 N15855-0 N21519-0 N18698-0 N24176-0 N42961-0 N53044-0 N60272-0 N61623-0 N57713-0 N14713-0 N47067-0 N29001-0 N46751-0 N32087-0 N2-0 N56517-0 N32387-0 N9240-0 N7754-0 N54482-0 N26376-0 N8957-0 N53214-0 N60858-0 N3894-0 N35272-0 N48890-0 N43299-0 N59436-0 N62688-0 N63538-0 N57368-0 N555-0 N47981-0 N58410-0 N25587-0 N20954-0 N59343-0 N45723-0 N43789-0 N52622-0 N46878-0 N39461-0 N12280-0 N15761-0 N3017

In [0]:
behaviors = behaviors.withColumn(
    "Timestamp", 
    unix_timestamp("Timestamp", "MM/d/yyyy h:mm:ss a").cast("timestamp")
)

In [0]:
def remove_duplicates(history):
    if isinstance(history, str):
        return ' '.join(sorted(set(history.split())))
    return history

remove_duplicates_udf = udf(remove_duplicates, StringType())
behaviors = behaviors.withColumn("History", remove_duplicates_udf("History"))
behaviors = behaviors.withColumn("History", when(col("History").isNotNull(), col("History")).otherwise(""))
behaviors = behaviors.dropDuplicates()

display(behaviors)

Impression ID,User ID,Timestamp,History,Impressions
88,U69950,2019-11-14T10:57:37Z,N10347 N11282 N12194 N16304 N18094 N18360 N18881 N20575 N20649 N2083 N21087 N21771 N2309 N26642 N27377 N27468 N28296 N31801 N32421 N32545 N34947 N35802 N38298 N39798 N40692 N43142 N43195 N43391 N46846 N4717 N48715 N51706 N58090 N5905 N60050 N61388 N6233 N65123 N8200 N9172 N9674,N10960-0 N61296-0 N6578-0 N52554-0 N62318-0 N4021-0 N53515-0 N44698-0 N6477-0 N41165-0 N35815-0 N23446-0 N39949-0 N60186-0 N25165-0 N37994-0 N62563-0 N6837-0 N46821-0 N29212-0 N20413-0 N45523-0 N20676-0 N50872-0 N40326-0 N38779-0 N15043-0 N61595-0 N15279-0 N1539-1 N36251-0 N38797-0 N23547-0 N8907-0 N58814-0
98,U47761,2019-11-13T06:07:16Z,N11821 N16384 N19079 N22479 N27642 N28614 N31225 N32089 N37075 N46033 N49647 N53074 N53354 N53520 N53526 N55576 N55846 N56460 N6441 N872,N9734-0 N25949-0 N47061-0 N14726-0 N59272-0 N36659-0 N41122-0 N20076-0 N35047-0 N24157-0 N13907-0 N35233-0 N14592-0 N9623-0 N8807-0 N24423-0 N6857-0 N4201-0 N36184-0 N43386-0 N42977-0 N4754-0 N47149-0 N13579-0 N34612-0 N17586-0 N61571-0 N15279-0 N55050-0 N55281-0 N19592-0 N17759-0 N42143-0 N45266-0 N20495-0 N43102-0 N48063-0 N38159-0 N35170-0 N19444-0 N7121-0 N51853-0 N849-0 N29468-0 N62128-0 N47717-0 N18312-1 N34876-0 N40468-0 N11869-0 N64734-0 N57426-0 N59673-0 N28213-0 N59469-0 N25881-0 N60992-0 N41172-0
284,U10932,2019-11-12T13:47:18Z,N17589 N29802 N35022 N3560 N36270 N49481 N50475 N50999,N20015-0 N45389-0 N62386-0 N22339-0 N21428-0 N33212-0 N13974-0 N6975-0 N60105-0 N31370-0 N12424-0 N51355-0 N21753-0 N3344-0 N12493-0 N30475-0 N7419-0 N52485-0 N21741-0 N62360-0 N18708-0 N40839-0 N55606-0 N64851-0 N2823-0 N58363-0 N16096-0 N44422-0 N6056-0 N25624-0 N32182-0 N48046-0 N26639-0 N18887-0 N32010-0 N61408-0 N63970-0 N3123-0 N36516-0 N59469-0 N45422-1
408,U13674,2019-11-13T15:18:02Z,N12349 N17109 N25739 N28030 N29068 N31801 N36517 N41616 N54095 N56630 N64738 N8485,N36638-0 N56214-1 N45509-0 N26376-0 N11769-0 N20576-0 N7618-0 N55505-0 N17529-0 N9354-0 N35958-0 N58114-0 N25064-0 N47231-0 N41612-0 N64096-0 N27869-0 N27540-0 N35767-0 N4642-0 N55132-0 N16291-0 N55281-0 N63106-0 N50014-0 N34048-0 N51287-0 N42457-0 N63656-0
770,U38627,2019-11-14T04:48:10Z,N21136 N2309 N26026 N306 N36530,N6816-0 N4404-0 N40559-0 N42457-0 N30089-0 N8015-0 N48017-0 N23749-0 N16148-0 N30071-0 N64174-0 N21701-0 N20603-0 N16419-0 N45742-0 N33831-0 N26673-0 N9623-0 N59138-0 N14436-0 N42868-0 N60550-0 N4021-0 N48937-0 N48077-0 N29133-0 N7121-0 N31679-0 N58086-0 N9135-0 N50872-0 N3957-0 N10960-1 N28767-0 N47098-0 N32854-0 N38215-0 N64412-0 N29739-0 N36016-0 N23877-0 N16777-0 N61483-0 N59214-0 N60272-0 N41165-0 N13981-0 N61787-0 N27869-0 N45509-0 N40109-0 N16844-0 N1012-0 N49712-0 N23864-0 N11817-0
892,U78483,2019-11-14T06:04:10Z,N10359 N11246 N12732 N14349 N15288 N18760 N18870 N19594 N1985 N2003 N20268 N26319 N27268 N3046 N30765 N33132 N37533 N37814 N39798 N41244 N41651 N434 N43955 N44867 N45887 N46239 N47479 N50710 N53880 N54571 N55556 N55897 N57011 N60702 N61471 N63062 N63971,N61623-0 N11817-0 N38490-0 N64228-0 N8907-0 N62227-0 N29133-0 N64412-0 N43587-0 N57451-0 N47098-0 N23877-0 N45509-0 N9621-0 N18862-0 N16419-0 N62933-0 N45317-0 N3167-0 N16844-0 N23008-0 N33576-0 N31395-0 N40676-0 N62750-0 N20770-0 N45428-0 N50107-0 N51552-0 N26262-0 N41612-0 N25160-0 N23814-0 N31679-0 N18423-0 N8015-0 N11269-0 N48077-0 N21519-0 N7754-0 N1999-0 N50389-0 N19300-0 N8509-0 N43460-0 N3841-0 N45734-0 N19195-0 N39389-0 N55488-0 N58814-0 N38442-0 N50601-0 N60186-0 N59385-0 N48017-0 N29536-0 N19444-0 N40559-0 N59214-0 N55555-0 N9623-0 N41172-0 N33831-1 N37793-0 N1319-0 N51187-0 N25163-0 N10960-0 N11940-0 N23805-0 N61483-0 N50619-0 N14780-0 N29739-0 N23824-0 N58660-0 N64174-0 N6056-0 N496-0 N4754-0 N50872-0 N55275-0 N49483-0 N10552-0 N64634-0 N34600-0 N4404-0 N23547-0 N59761-0 N13094-0 N62563-0 N20793-0 N13689-0
1296,U36330,2019-11-12T12:46:59Z,N16636 N18285 N2203 N27448 N33976 N42620 N45971,N9719-0 N40839-0 N34998-0 N1555-0 N63970-1 N57426-0 N6975-0 N31029-0

In [0]:
behaviors.coalesce(1).write.mode("overwrite").parquet("mnt/newsrecsysdata/transformed-data/behaviors")